In [1]:
import os
import json
import copy
from pathlib import Path
from datetime import datetime, timezone
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from anthropic import Anthropic

ROOT = Path.cwd()
if not (ROOT / "notebooks").exists() and (ROOT.parent / "notebooks").exists():
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")
load_dotenv()

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")

print("Model:", MODEL)

Model: claude-haiku-4-5-20251001


In [2]:
PROFILE_SCHEMA = {
    "transaction_type": None,       # sale, rent, or None
    "property_type": [],            # apartment, villa, townhouse, etc.
    "hard_constraints": {
        "budget_min": None,
        "budget_max": None,
        "bedrooms_min": None,
        "bedrooms_max": None,
        "bathrooms_min": None,
        "communities": [],
        "required_features": [],
        "excluded_features": [],
        "other": [],
    },
    "intent_preferences": {
        "Family-Friendly": [],
        "Investment": [],
        "Lifestyle": [],
        "Views": [],
    },
    "soft_preferences": {
        "preferred_communities": [],
        "preferred_property_types": [],
        "amenities": [],
        "other": [],
    },
    "updated_at": None,
}


def new_customer_profile(customer_id: str) -> dict:
    """Create an empty profile for a customer."""
    return {
        "customer_id": str(customer_id),
        "profile": copy.deepcopy(PROFILE_SCHEMA),
        "version": 1,
    }


def normalize_text_list(value):
    """Normalize a list of text values."""
    if value is None:
        return []

    if isinstance(value, str):
        value = [value]

    if not isinstance(value, list):
        return []

    result = []
    for item in value:
        if isinstance(item, str) and item.strip():
            cleaned = item.strip()
            if cleaned not in result:
                result.append(cleaned)

    return result

In [3]:
QUERY_PLAN_SCHEMA = {
    "query_type": "property_search",
    "transaction_type": None,
    "property_types": [],
    "hard_constraints": {
        "budget_min": None,
        "budget_max": None,
        "bedrooms_min": None,
        "bedrooms_max": None,
        "bathrooms_min": None,
        "communities": [],
        "required_features": [],
        "excluded_features": [],
        "other": [],
    },
    "intent_preferences": {
        "Family-Friendly": [],
        "Investment": [],
        "Lifestyle": [],
        "Views": [],
    },
    "soft_preferences": {
        "preferred_communities": [],
        "preferred_property_types": [],
        "amenities": [],
        "other": [],
    },
    "semantic_query": "",
    "retrieval_methods": [
        "milvus_vector",
        "neo4j_graph",
    ],
    "clarification_needed": False,
    "clarification_question": None,
}

In [4]:

INTENT_NAMES = ["Family-Friendly", "Investment", "Lifestyle", "Views"]

def string_list_schema():
    return {
        "type": "array",
        "items": {"type": "string"},
    }

def object_schema(properties, required=None):
    schema = {
        "type": "object",
        "properties": properties,
    }
    if required:
        schema["required"] = required
    return schema

hard_fields = {
    "budget_min": {"type": "number"},
    "budget_max": {"type": "number"},
    "bedrooms_min": {"type": "integer"},
    "bedrooms_max": {"type": "integer"},
    "bathrooms_min": {"type": "integer"},
    "communities": string_list_schema(),
    "required_features": string_list_schema(),
    "excluded_features": string_list_schema(),
    "other": string_list_schema(),
}

intent_fields = {name: string_list_schema() for name in INTENT_NAMES}

soft_fields = {
    name: string_list_schema()
    for name in [
        "preferred_communities",
        "preferred_property_types",
        "amenities",
        "other",
    ]
}

plan_schema = object_schema({
    "query_type": {
        "type": "string",
        "enum": [
            "property_search",
            "property_question",
            "market_research",
            "other",
        ],
    },
    "transaction_type": {
        "type": "string",
        "enum": ["sale", "rent", "unknown"],
    },
    "property_types": string_list_schema(),
    "hard_constraints": object_schema(hard_fields),
    "intent_preferences": object_schema(intent_fields),
    "soft_preferences": object_schema(soft_fields),
    "semantic_query": {"type": "string"},
    "retrieval_methods": {
        "type": "array",
        "items": {
            "type": "string",
            "enum": ["milvus_vector", "neo4j_graph", "hard_filter"],
        },
    },
    "clarification_needed": {"type": "boolean"},
    "clarification_question": {"type": "string"},
})

# Profile updates: omitted fields mean "unchanged".
# Empty lists explicitly clear list fields.
# Use clear_fields to clear scalar fields.
profile_update_schema = object_schema({
    "transaction_type": {
        "type": "string",
        "enum": ["sale", "rent"],
    },
    "property_type": string_list_schema(),
    "hard_constraints": object_schema(hard_fields),
    "intent_preferences": object_schema(intent_fields),
    "soft_preferences": object_schema(soft_fields),
    "clear_fields": string_list_schema(),
})

CLASSIFICATION_TOOL = {
    "name": "classify_customer_query",
    "description": (
        "Classify the latest real-estate query and propose changes "
        "to the existing customer profile in one call. Omit unchanged "
        "profile fields. Use clear_fields to remove saved values."
    ),
    "input_schema": object_schema(
        {
            "query_plan": plan_schema,
            "profile_updates": profile_update_schema,
        },
        required=["query_plan", "profile_updates"],
    ),
}

print("Classification tool schema rebuilt.")

Classification tool schema rebuilt.


In [5]:
SYSTEM_PROMPT = """
You are a real-estate customer-query classifier.

Return BOTH:
1. query_plan: a structured plan for the current request.
2. profile_updates: only new or changed information about the customer.

CUSTOMER PROFILE AND CONVERSATION CONTEXT

Use the existing customer profile and recent conversation context to
interpret the latest message.

1. Extract explicit requirements: transaction type, property type, budget,
   bedrooms, bathrooms, communities, required features, and excluded features.

2. Classify supported intent preferences into:
   Family-Friendly, Investment, Lifestyle, and Views.

3. Separate hard constraints from soft preferences.
   Hard constraints are mandatory. Soft preferences influence ranking but
   must not automatically exclude otherwise suitable listings.

4. Merge new information with existing profile information.
   Preserve existing preferences unless the customer changes, removes,
   or contradicts them.

5. Corrections replace old values. For example, "increase my budget to
   2.5 million" replaces the previous maximum budget.

6. Do not invent requirements, budget amounts, locations, or preferences.
   Use null for unknown scalar values and empty lists when no values apply.

7. In profile_updates, null means "leave unchanged".
   Use clear_fields to explicitly remove an existing value.
   An empty list means "replace the saved list with an empty list".

8. Do not save a one-time search condition as a permanent preference unless
   the customer expresses it as a continuing preference.

9. Use the current query and relevant existing profile preferences to
   construct query_plan. The current request takes priority if it conflicts
   with an older preference.

10. Use the four supported intent categories only. Do not invent new
    intent categories.

11. Keep the semantic_query natural and useful for vector retrieval.
    Do not include unsupported facts.

12. Use retrieval_methods from the allowed options. Do not claim a retrieval
    method has already run.

13. If an essential detail is ambiguous, set clarification_needed and
    provide one concise clarification_question. Still return the best
    possible partial plan.

14. Keep all money amounts in the currency/unit expressed or established
    by the conversation. Never silently convert currencies.

15. Do not treat every mentioned feature as mandatory. Preserve the
    customer's actual wording and intent.
"""

In [6]:
def get_nested(data: dict, path: str):
    current = data
    for key in path.split("."):
        if not isinstance(current, dict) or key not in current:
            return None
        current = current[key]
    return current


def set_nested(data: dict, path: str, value: Any):
    keys = path.split(".")
    current = data

    for key in keys[:-1]:
        if key not in current or not isinstance(current[key], dict):
            raise ValueError(f"Invalid profile path: {path}")
        current = current[key]

    if keys[-1] not in current:
        raise ValueError(f"Unknown profile field: {path}")

    current[keys[-1]] = copy.deepcopy(value)


def merge_profile(profile: dict, patch: dict) -> dict:
    """Apply a validated profile patch without erasing unrelated fields."""
    updated = copy.deepcopy(profile)

    allowed_paths = {
        "transaction_type",
        "property_type",
        *[
            f"hard_constraints.{field}"
            for field in PROFILE_SCHEMA["hard_constraints"]
        ],
        *[
            f"intent_preferences.{field}"
            for field in PROFILE_SCHEMA["intent_preferences"]
        ],
        *[
            f"soft_preferences.{field}"
            for field in PROFILE_SCHEMA["soft_preferences"]
        ],
    }

    def apply_patch(target, changes, prefix=""):
        if not isinstance(changes, dict):
            return

        for key, value in changes.items():
            path = f"{prefix}.{key}" if prefix else key

            if path not in allowed_paths or value is None:
                continue

            if isinstance(value, dict):
                apply_patch(target, value, path)
            else:
                set_nested(target, path, value)

    apply_patch(updated, patch)

    for path in patch.get("clear_fields", []):
        if path not in allowed_paths:
            raise ValueError(f"Refusing to clear unknown profile field: {path}")

        current = get_nested(updated, path)

        if isinstance(current, list):
            set_nested(updated, path, [])
        else:
            set_nested(updated, path, None)

    updated["updated_at"] = datetime.now(timezone.utc).isoformat()
    return updated

In [8]:
INTENT_CATEGORIES = {
    "Family-Friendly",
    "Investment",
    "Lifestyle",
    "Views",
}

ALLOWED_RETRIEVAL_METHODS = {
    "milvus_vector",
    "neo4j_graph",
    "hard_filter",
}


def validate_classification(result: dict) -> dict:
    if not isinstance(result, dict):
        raise ValueError("Classifier output must be a dictionary.")

    plan = result.get("query_plan")
    patch = result.get("profile_updates")

    if not isinstance(plan, dict) or not isinstance(patch, dict):
        raise ValueError("Missing query_plan or profile_updates.")

    if plan.get("transaction_type") not in (None, "sale", "rent"):
        raise ValueError("Invalid transaction_type.")

    if plan.get("query_type") not in {
        "property_search", "property_question", "market_research", "other"
    }:
        raise ValueError("Invalid query_type.")

    if not isinstance(plan.get("hard_constraints"), dict):
        raise ValueError("Invalid hard_constraints.")

    if not isinstance(plan.get("intent_preferences"), dict):
        raise ValueError("Invalid intent_preferences.")

    if not set(plan["intent_preferences"]).issubset(INTENT_CATEGORIES):
        raise ValueError("Unexpected intent category.")

    if not set(plan.get("retrieval_methods", [])).issubset(
        ALLOWED_RETRIEVAL_METHODS
    ):
        raise ValueError("Unexpected retrieval method.")

    if not isinstance(patch.get("clear_fields", []), list):
        raise ValueError("clear_fields must be a list.")

    return result

In [9]:
def classify_customer_query(
    customer_id: str,
    latest_query: str,
    customer_profile: dict | None = None,
    conversation_history: list | None = None,
) -> dict:
    """
    Return the current query plan and an updated customer profile.

    The profile is passed in by the caller; this function does not
    persist it to a database.
    """
    if not latest_query or not latest_query.strip():
        raise ValueError("latest_query cannot be empty.")

    if customer_profile is None:
        customer_profile = new_customer_profile(customer_id)

    existing_profile = customer_profile.get(
        "profile", copy.deepcopy(PROFILE_SCHEMA)
    )

    context = {
        "customer_id": str(customer_id),
        "existing_profile": existing_profile,
        "conversation_history": (conversation_history or [])[-10:],
        "latest_query": latest_query.strip(),
    }

    response = client.messages.create(
        model=MODEL,
        max_tokens=1800,
        system=SYSTEM_PROMPT,
        tools=[CLASSIFICATION_TOOL],
        tool_choice={
            "type": "tool",
            "name": "classify_customer_query",
        },
        messages=[
            {
                "role": "user",
                "content": json.dumps(context, ensure_ascii=False, default=str),
            }
        ],
    )

    tool_result = next(
        (
            block.input
            for block in response.content
            if block.type == "tool_use"
            and block.name == "classify_customer_query"
        ),
        None,
    )

    if not isinstance(tool_result, dict):
        raise ValueError("The model did not return the expected tool output.")

    result = validate_classification(tool_result)

    # Normalize unknown values for the rest of the Python pipeline.
    plan = result["query_plan"]

    if plan.get("transaction_type") == "unknown":
        plan["transaction_type"] = None

    if not plan.get("clarification_question", "").strip():
        plan["clarification_question"] = None

    for field in [
        "budget_min", "budget_max",
        "bedrooms_min", "bedrooms_max", "bathrooms_min",
    ]:
        plan["hard_constraints"].setdefault(field, None)

    for field in [
        "communities", "required_features", "excluded_features", "other",
    ]:
        plan["hard_constraints"].setdefault(field, [])

    for name in INTENT_NAMES:
        plan["intent_preferences"].setdefault(name, [])

    for field in [
        "preferred_communities", "preferred_property_types",
        "amenities", "other",
    ]:
        plan["soft_preferences"].setdefault(field, [])

    result["query_plan"] = plan

    # Apply the patch in Python, not inside the LLM.
    updated_profile = merge_profile(
        existing_profile,
        result["profile_updates"],
    )

    updated_customer = {
        **customer_profile,
        "customer_id": str(customer_id),
        "profile": updated_profile,
        "version": customer_profile.get("version", 1) + 1,
    }

    return {
        "customer_id": str(customer_id),
        "query_plan": result["query_plan"],
        "profile_updates": result["profile_updates"],
        "customer_profile": updated_customer,
        "model": MODEL,
    }

In [10]:
customer = new_customer_profile("customer_001")

first_result = classify_customer_query(
    customer_id="customer_001",
    latest_query=(
        "I'm looking to buy a 2-bedroom apartment in Dubai Marina "
        "for no more than 2 million. I'd prefer sea views and good "
        "rental potential."
    ),
    customer_profile=customer,
)

customer = first_result["customer_profile"]

print("QUERY PLAN")
print(json.dumps(first_result["query_plan"], indent=2, ensure_ascii=False))

print("\nSAVED CUSTOMER PROFILE")
print(json.dumps(customer["profile"], indent=2, ensure_ascii=False))

QUERY PLAN
{
  "query_type": "property_search",
  "transaction_type": "sale",
  "property_types": [
    "apartment"
  ],
  "semantic_query": "2-bedroom apartment in Dubai Marina with sea views and rental potential",
  "hard_constraints": {
    "bedrooms_min": 2,
    "bedrooms_max": 2,
    "budget_max": 2000000,
    "communities": [
      "Dubai Marina"
    ],
    "required_features": [],
    "budget_min": null,
    "bathrooms_min": null,
    "excluded_features": [],
    "other": []
  },
  "soft_preferences": {
    "preferred_communities": [
      "Dubai Marina"
    ],
    "preferred_property_types": [
      "apartment"
    ],
    "other": [],
    "amenities": []
  },
  "intent_preferences": {
    "Views": [
      "sea views"
    ],
    "Investment": [
      "good rental potential"
    ],
    "Family-Friendly": [],
    "Lifestyle": []
  },
  "retrieval_methods": [
    "hard_filter",
    "milvus_vector"
  ],
  "clarification_needed": false,
  "clarification_question": null
}

SAVED CUSTO

In [11]:
second_result = classify_customer_query(
    customer_id="customer_001",
    latest_query=(
        "Actually, increase my budget to 2.5 million, "
        "but I still want two bedrooms and sea views."
    ),
    customer_profile=customer,
    conversation_history=[
        {
            "role": "user",
            "content": (
                "I want to buy a 2-bedroom apartment in Dubai Marina "
                "for up to 2 million, preferably with sea views."
            ),
        }
    ],
)

customer = second_result["customer_profile"]

print("UPDATED QUERY PLAN")
print(json.dumps(second_result["query_plan"], indent=2, ensure_ascii=False))

print("\nUPDATED PROFILE")
print(json.dumps(customer["profile"], indent=2, ensure_ascii=False))

UPDATED QUERY PLAN
{
  "query_type": "property_search",
  "transaction_type": "sale",
  "property_types": [
    "apartment"
  ],
  "semantic_query": "2-bedroom apartment in Dubai Marina with sea views",
  "hard_constraints": {
    "bedrooms_min": 2,
    "bedrooms_max": 2,
    "budget_max": 2500000,
    "communities": [
      "Dubai Marina"
    ],
    "required_features": [
      "sea views"
    ],
    "budget_min": null,
    "bathrooms_min": null,
    "excluded_features": [],
    "other": []
  },
  "intent_preferences": {
    "Views": [
      "sea views"
    ],
    "Family-Friendly": [],
    "Investment": [],
    "Lifestyle": []
  },
  "soft_preferences": {
    "preferred_communities": [
      "Dubai Marina"
    ],
    "preferred_property_types": [],
    "amenities": [],
    "other": []
  },
  "retrieval_methods": [
    "hard_filter",
    "milvus_vector"
  ],
  "clarification_needed": false,
  "clarification_question": null
}

UPDATED PROFILE
{
  "transaction_type": "sale",
  "property

In [12]:
plan = second_result["query_plan"]

summary = {
    "Query type": plan["query_type"],
    "Transaction": plan["transaction_type"],
    "Property types": plan["property_types"],
    "Hard constraints": plan["hard_constraints"],
    "Intent preferences": plan["intent_preferences"],
    "Soft preferences": plan["soft_preferences"],
    "Semantic query": plan["semantic_query"],
    "Retrieval methods": plan["retrieval_methods"],
    "Clarification needed": plan["clarification_needed"],
    "Clarification question": plan["clarification_question"],
}

pd.DataFrame(
    [{"field": key, "value": json.dumps(value, ensure_ascii=False)} 
     for key, value in summary.items()]
)

,field,value
0,Query type,"""property_search"""
1,Transaction,"""sale"""
2,Property types,"[""apartment""]"
3,Hard constraints,"{""bedrooms_min"": 2, ""bedrooms_max"": 2, ""budget..."
4,Intent preferences,"{""Views"": [""sea views""], ""Family-Friendly"": []..."
5,Soft preferences,"{""preferred_communities"": [""Dubai Marina""], ""p..."
6,Semantic query,"""2-bedroom apartment in Dubai Marina with sea ..."
7,Retrieval methods,"[""hard_filter"", ""milvus_vector""]"
8,Clarification needed,false
9,Clarification question,null
